# Random loss-screened initialization benchmark
Compare zero initialization with **1, 5, 10, 20, 100 random starting points** on representative saved PD cases. Each pool also includes zero. Candidates are scored with the existing **loss only** (the object is refitted, but no phase gradient/Hessian is evaluated); then exactly one Newton optimization starts from the lowest-loss candidate. This is screening initializations, not optimizing every candidate.

The default benchmark replays three stacks from each archived AO trajectory. Strategies see identical images and the same previous SLM command, isolating the initializer. A replay result is the hypothetical residual after replacing that loop's update; it is **not** a cumulative trajectory for the new strategy. An optional final section runs actual three-loop feedback trajectories with fresh acquisitions between corrections.

Nested candidate pools reuse losses. Identical selected starts reuse the optimized result. A better initial loss does not guarantee a better final minimum. Screening still includes PSF propagation and sample SVD solves, so its cost is measured separately.

Exposure calibration and the three-mode joint diversity scheme stay as in the completed sweep. Ground truth is used only to replay simulation and report RMS, never to select starts. Checkpoints and image caches are separate from the original sweep data.

In [ ]:
from pathlib import Path
import sys, json, hashlib, os, tempfile, time
ROOT = Path.cwd()
while not ((ROOT / "utils").is_dir() and (ROOT / "optimization").is_dir()):
    if ROOT == ROOT.parent:
        raise RuntimeError("Run this notebook inside the repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
EXPERIMENT_DIR = ROOT / "systematic_testing" / "random_start_benchmark"
FIGURES_DIR = EXPERIMENT_DIR / "figures"
RESULTS_DIR = EXPERIMENT_DIR / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from utils import psf, plotting, zernike
from optimization import reconstruction
from systematic_testing import correction

In [ ]:
plt.style.use("bmh")
plt.rcParams["figure.figsize"] = [548 / 72, 390 / 72]
plt.rcParams["font.size"] = 12
plt.rcParams['text.usetex'] = True
plt.rcParams["figure.dpi"] = 300

In [ ]:
SOURCE_PATH = ROOT / 'systematic_testing/rms_snr_sweep/results/pd_rms_snr_sweep_results_large_sweep.npz'
source = np.load(SOURCE_PATH, allow_pickle=False)
# Archive indices are zero-based. Two RMS=0.15 rows have different noise seeds.
CASE_INDICES = [
    #(0, 6, 0),  # Success: .05 RMS, SNR100.
    #(4, 6, 1),  # Success: .20 RMS, SNR100.
    #(0, 1, 0),  # Small-error later degradation: .05 RMS, SNR10.
    (5, 3, 2),  # Strong later degradation: .25 RMS, SNR20.
    (7, 5, 2),  # Strong later degradation: .40 RMS, SNR50.
    (2, 6, 2),  # Confirmed first-loop wrong basin: .10 RMS, SNR100.
]
RANDOM_START_COUNTS = [0, 50, 100, 1000]  # 0 = zero-only baseline; others add this many random candidates.
START_RMS_RANGE = (0.005, 0.50)  # Log-uniform candidate RMS in waves; independent of truth.
START_SEED = 17
BACKEND = 'auto'
SCORING_CPU_WORKERS = 1
PRINT_OPTIMIZER_STEPS = True
RUN_REPLAY_BENCHMARK = True
RUN_CLOSED_LOOP = False  # Optional; more expensive, because trajectories differ between strategies.
OUTPUT_DIR = ROOT / 'systematic_testing/random_start_benchmark/results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / 'replay_results.json'
CLOSED_LOOP_PATH = OUTPUT_DIR / 'closed_loop_results.json'
backend = reconstruction.get_backend(BACKEND)
if backend == 'cuda' and SCORING_CPU_WORKERS != 1:
    raise ValueError('Use SCORING_CPU_WORKERS=1 for CUDA.')
counts = sorted(set(int(n) for n in RANDOM_START_COUNTS))
if not counts or counts[0] < 0 or any(int(n) != n for n in RANDOM_START_COUNTS):
    raise ValueError('Candidate counts must be nonnegative integers.')
if not 0 < START_RMS_RANGE[0] <= START_RMS_RANGE[1]:
    raise ValueError('Require 0 < lower RMS <= upper RMS.')
for indices in CASE_INDICES:
    if not np.all(source['completed'][indices]):
        raise ValueError(f'Incomplete archived case: {indices}')
for i, j, t in CASE_INDICES:
    print(f'Case {(i,j,t)} | initial RMS {source["initial_rms"][i]:g}, '
          f'SNR {source["reference_snr"][j]:g}, trial {t+1} | '
          f'archived RMS {np.round(source["residual_rms"][i,j,t], 5)}')

In [ ]:
N_order = 3                # 3 for 3P, 2 for 2P
lambd = 1.3e-3             # Wavelength [mm]
n = 1.333                  # Refractive index
k = 2*n*np.pi/lambd        # Wavenumber
num_apt = 1.05             # Numerical aperture
f = 7.2                    # Focal length of objective (Olympus) [mm]
grid_size = 128            # BFP resolution
L_bfp = 15.12              # BFP diameter [mm]
mag = 4                    # Magnification rate from input to objective
w_0 = 3.5                  # mm
num_apt = 1.05
L_ffp = 0.0065             # FFP size [mm]
grid_ffp = 32
grid_bfp = 32
microscope = psf.Microscope(N_order = N_order,
                            lambd = lambd,
                            n = n,
                            num_apt = num_apt,
                            f = f,
                            mag = mag,
                            w_0 = w_0,
                            L_bfp = L_bfp,
                            grid_bfp = grid_bfp)

grid = psf.Centered_Square_Grid(L_ffp = L_ffp,
                                grid_ffp = grid_ffp,
                                z_level = 0)

xs, ys = grid.get_xy()


modes = [[-2,2],
         [2,2],
         [-3,3],
         [-1,3],
         [1,3],
         [3,3],
         [-4,4],
         [-2,4],
         [0,4],
         [2,4],
         [4,4],
         [-5,5],
         [-3, 5],
         [-1, 5],
         [1, 5],
         [3, 5],
         [5,5]]
# The completed sweep used the current 32x32 optics; verify replay calibration.
modes = source['modes'].tolist()
sample = psf.bead_img_3D(grid, source['sample_z'], [0], [0], [0], [.001])
reference = microscope.compute_image_3D(sample, zernike.EmptyAberration(), [0.0], mode='vector')[3]
for snr, exposure in zip(source['reference_snr'], source['exposures']):
    np.testing.assert_allclose(correction.exposure_for_snr(reference, float(snr)), exposure,
                               rtol=1e-10, err_msg='Optics/sample differ from saved sweep; fix setup before replay.')
OPTIMIZER_OPTIONS = dict(method='newton', max_iterations=int(source['max_optimizer_iterations']), verbose=False)

In [ ]:
# Fingerprinted checkpoints prevent accidental mixing of different experiments.
source_fingerprint = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
config = dict(source_sha256=source_fingerprint, cases=[list(c) for c in CASE_INDICES],
              random_start_counts=counts, start_rms_range=list(START_RMS_RANGE), start_seed=START_SEED,
              backend=backend, optimizer_options=OPTIMIZER_OPTIONS,
              grid_shape=[grid.grid_ffp_x, grid.grid_ffp_y], pupil_size=microscope.grid_bfp)

def atomic_json(path, value):
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', dir=path.parent, prefix=path.name+'.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            json.dump(value, stream, indent=2, allow_nan=False)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None: temporary.unlink(missing_ok=True)

if CHECKPOINT_PATH.exists():
    replay_state = json.loads(CHECKPOINT_PATH.read_text())
    if replay_state['config'] != config:
        raise ValueError('Checkpoint configuration differs. Choose a new OUTPUT_DIR for a new experiment.')
else:
    replay_state = dict(config=config, stacks={})

def make_stack_geometry(loop):
    original = source['diversity_strengths']
    biases = np.sign(original)*np.maximum(abs(original)*float(source['diversity_decay'])**loop,
                                         np.minimum(abs(original), float(source['min_diversity_strength'])))
    diversities = [zernike.Aberration([m], [float(b)]) for _ in source['focal_positions']
                   for m in source['diversity_modes'] for b in biases]
    focal = np.repeat(source['focal_positions'], len(source['diversity_modes'])*len(biases))
    return focal, diversities

def candidate_pool(case, loop):
    # Each candidate draws its direction and radius together: increasing max count preserves prefixes.
    rng = np.random.default_rng(np.random.SeedSequence([START_SEED, *case, loop]))
    pool = [np.zeros(len(modes))]
    for _ in range(max(counts)):
        direction = rng.normal(size=len(modes))
        radius = np.exp(rng.uniform(np.log(START_RMS_RANGE[0]), np.log(START_RMS_RANGE[1])))
        unit_rms = zernike.zernike_RMS(zernike.Aberration(modes, direction), microscope.alpha)
        pool.append(direction*radius/unit_rms)
    return np.array(pool)

def make_workspace(images, loop):
    focal, diversities = make_stack_geometry(loop)
    return reconstruction.ReconstructionWorkspace(
        microscope, grid, images, focal, source['reconstruction_z'], modes, float(source['rho']),
        diversities=diversities, mode='vector', backend=backend, reuse_svd=False)


def optimize_selected(workspace, start, prior_slm, residual_truth, label):
    # Clear loss-only cached SVD factors before retaining factors for the exact Hessian.
    workspace.clear_cache()
    workspace.reuse_svd = True
    def report(event):
        if PRINT_OPTIMIZER_STEPS:
            error = zernike.zernike_RMS_difference(zernike.Aberration(modes, event['strengths']),
                                                   residual_truth, microscope.alpha)
            print(f"{label}: step {event['iteration']}, loss={event['loss']:.6e}, RMS error={error:.6f}", flush=True)
    started = time.perf_counter()
    estimated, info = workspace.optimize(start, on_iteration=report,
                                         **dict(OPTIMIZER_OPTIONS, num_random_starts=0))
    elapsed = time.perf_counter()-started
    update = -float(source['correction_gain'])*np.clip(
        estimated.strengths, -float(source['max_modal_update']), float(source['max_modal_update']))
    after = residual_truth + zernike.Aberration(modes, update)
    result = dict(residual_rms=float(zernike.zernike_RMS(after, microscope.alpha)),
                  slm_strengths=(prior_slm+update).tolist(), estimated_strengths=estimated.strengths.tolist(),
                  loss=float(info['loss']), steps=int(info['iterations']), success=bool(info['success']),
                  message=info['message'], optimization_seconds=elapsed)
    workspace.clear_cache()
    workspace.reuse_svd = False
    return result

## Replay benchmark: identical images for every initialization strategy
Cache each case's three original stacks using the archived noise seed and accumulated commands. The original zero-start results are used as the baseline, avoiding redundant full optimizations. If zero wins a random candidate pool, its archived result is reused. The best **initial** loss is selected without looking at the final RMS.

Each plot below compares a replacement update on the archived prior state, not a new cumulative AO trajectory. The optional closed-loop section is needed to measure the actual combined effect of three updates.

In [ ]:
# Cache original trajectories once; caching preserves fresh, independent noise between AO loops.
replay_stacks = {}
for case in CASE_INDICES:
    i, j, t = case
    cache_path = OUTPUT_DIR / f'images_{i}_{j}_{t}.npz'
    if cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as cached:
            if str(cached['source_sha256']) != source_fingerprint:
                raise ValueError('Image cache source differs; use a new output directory.')
            stacks = cached['images'].copy()
    else:
        truth_case = zernike.Aberration(modes, source['truth_strengths'][i,t])
        simulator = correction.SimulatedSLM(
            microscope, sample, truth_case, modes, float(source['exposures'][j]),
            np.random.default_rng(np.random.SeedSequence([int(source['noise_seed']), i,j,t])))
        stacks = []
        for loop in range(3):
            prior = np.zeros(len(modes)) if loop == 0 else source['slm_strengths'][i,j,t,loop-1]
            simulator.set_slm(prior)
            focal, diversities = make_stack_geometry(loop)
            stacks.append(simulator.acquire(focal, diversities))
        stacks = np.array(stacks)
        with tempfile.NamedTemporaryFile(mode='wb', dir=OUTPUT_DIR, delete=False) as stream:
            temp = Path(stream.name)
            np.savez_compressed(stream, images=stacks, source_sha256=source_fingerprint)
        os.replace(temp, cache_path)
    replay_stacks[case] = stacks
print(f'Replay ready: {len(replay_stacks)*3} stacks. Sweep data is unchanged.')

In [ ]:
if RUN_REPLAY_BENCHMARK:
    for case in CASE_INDICES:
        i, j, t = case
        truth_case = zernike.Aberration(modes, source['truth_strengths'][i,t])
        for loop in range(3):
            key = f'{i},{j},{t},{loop}'
            record = replay_state['stacks'].setdefault(key, dict(candidate_losses=[], fits={}, results={}))
            if all(str(n) in record['results'] for n in counts):
                print(f'{key}: completed checkpoint reused')
                continue
            prior = np.zeros(len(modes)) if loop == 0 else source['slm_strengths'][i,j,t,loop-1]
            residual_truth = truth_case + zernike.Aberration(modes, prior)
            pool = candidate_pool(case, loop)
            workspace = make_workspace(replay_stacks[case][loop], loop)
            started = time.perf_counter()
            # No gradients/Hessians here. Checkpoint scores in small batches.
            done = len(record['candidate_losses'])
            with tqdm(total=len(pool), initial=done, desc=f'{key}: initial losses') as progress:
                for start in range(done, len(pool), 10):
                    started = time.perf_counter()
                    losses = workspace.losses(pool[start:start+10], workers=SCORING_CPU_WORKERS)
                    if not np.all(np.isfinite(losses)): raise FloatingPointError('Nonfinite candidate loss.')
                    record['candidate_losses'].extend(losses.tolist())
                    record['candidate_strengths'] = pool.tolist()
                    record['scoring_seconds'] = record.get('scoring_seconds', 0.0)+time.perf_counter()-started
                    atomic_json(CHECKPOINT_PATH, replay_state)
                    progress.update(len(losses))
            # Archived baseline applies the original clipped update and has exact saved residual RMS.
            record['fits'].setdefault('0', dict(
                residual_rms=float(source['residual_rms'][i,j,t,loop]),
                slm_strengths=source['slm_strengths'][i,j,t,loop].tolist(),
                loss=float(source['optimizer_loss'][i,j,t,loop]),
                steps=int(source['optimizer_steps'][i,j,t,loop]),
                success=bool(source['optimizer_success'][i,j,t,loop]),
                message='Archived zero-start fit', optimization_seconds=0.0))
            for n in counts:
                if str(n) in record['results']: continue
                selected = int(np.argmin(record['candidate_losses'][:n+1]))
                if str(selected) not in record['fits']:
                    record['fits'][str(selected)] = optimize_selected(
                        workspace, pool[selected], prior, residual_truth, f'{key}, {n} random')
                record['results'][str(n)] = dict(record['fits'][str(selected)], selected_index=selected,
                    initial_loss=record['candidate_losses'][selected], random_candidates=n,
                    initial_rms=float(zernike.zernike_RMS(zernike.Aberration(modes,pool[selected]),microscope.alpha)))
                atomic_json(CHECKPOINT_PATH,replay_state)
                result=record['results'][str(n)]
                print(f"{key}, {n} random + zero: selected #{selected}; residual={result['residual_rms']:.6f} waves",flush=True)
print(f'Replay checkpoint: {CHECKPOINT_PATH}')

In [ ]:
# Shared baseline stacks: each loop panel is an independent replacement-update comparison.
for case in CASE_INDICES:
    i,j,t = case
    fig, ax = plt.subplots(layout='constrained')
    for loop in range(3):
        record = replay_state['stacks'].get(f'{i},{j},{t},{loop}', {})
        values = [record.get('results',{}).get(str(n),{}).get('residual_rms',np.nan) for n in counts]
        ax.plot(np.arange(len(counts)), values, marker='o', label=f'Archived loop {loop+1} stack')
    ax.axhline(float(source['initial_rms'][i]),c='red',ls='dashed',label='Original uncorrected RMS')
    ax.set_xticks(np.arange(len(counts)), ['Zero']+[str(n) for n in counts if n != 0] if 0 in counts else [str(n) for n in counts])
    ax.set_xlabel('Number of random candidates screened (zero also included)')
    ax.set_ylabel('RMS after replacement correction [waves]')
    ax.set_title(f"RMS {source['initial_rms'][i]:g}, reference SNR {source['reference_snr'][j]:g}, trial {t+1}")
    ax.set_ylim(bottom=0)
    ax.legend()
    plt.show()
print('Replay results are conditional on the original prior corrections; use closed-loop tests below for trajectories.')

## Optional closed-loop test
Enable `RUN_CLOSED_LOOP` to test three actual AO loops per candidate-count strategy. Each new stack observes that strategy's own accumulated correction. One reproducible noise stream per strategy starts at the archived seed; subsequent Poisson samples may differ as image brightness changes. Noise equivalence is exact only when the optical states are identical. Loops 2 and 3 cannot share candidate losses across different trajectories.

Here zero starts are optimized normally, rather than taken from the archive. All strategies are checkpointed after completed loops, including the noise RNG state, so interrupted trajectories can resume. Completed replay loss screens can be reused for the common first-loop stack.

In [ ]:
if RUN_CLOSED_LOOP:
    if CLOSED_LOOP_PATH.exists():
        closed_state = json.loads(CLOSED_LOOP_PATH.read_text())
        if closed_state['config'] != config: raise ValueError('Closed-loop configuration changed; choose new output directory.')
    else:
        closed_state=dict(config=config, runs={})
    for case in CASE_INDICES:
        i,j,t=case
        truth_case=zernike.Aberration(modes,source['truth_strengths'][i,t])
        for n in counts:
            key=f'{i},{j},{t},{n}'
            history=closed_state['runs'].setdefault(key, [])
            if len(history)==3: continue
            simulator=correction.SimulatedSLM(
                microscope,sample,truth_case,modes,float(source['exposures'][j]),
                np.random.default_rng(np.random.SeedSequence([int(source['noise_seed']),i,j,t])))
            if history:
                simulator.set_slm(history[-1]['slm_strengths'])
                simulator.rng.bit_generator.state=history[-1]['noise_rng_state']
            for loop in range(len(history),3):
                focal,diversities=make_stack_geometry(loop)
                images=simulator.acquire(focal,diversities)
                workspace=make_workspace(images,loop)
                pool=candidate_pool(case,loop)[:n+1]
                replay=replay_state['stacks'].get(f'{i},{j},{t},0',{})
                started=time.perf_counter()
                if loop==0 and len(replay.get('candidate_losses',[]))>=n+1:
                    losses=np.array(replay['candidate_losses'][:n+1])
                else:
                    losses=workspace.losses(pool,workers=SCORING_CPU_WORKERS)
                if not np.all(np.isfinite(losses)):raise FloatingPointError('Nonfinite candidate loss.')
                screening_seconds=time.perf_counter()-started
                selected=int(np.argmin(losses))
                result=optimize_selected(workspace,pool[selected],simulator.slm_strengths,
                                        simulator._residual(), f'{key}, loop {loop+1}')
                simulator.set_slm(result['slm_strengths'])
                history.append(dict(result,selected_index=selected,initial_loss=float(losses[selected]),
                                    screening_seconds=screening_seconds,noise_rng_state=simulator.rng.bit_generator.state))
                atomic_json(CLOSED_LOOP_PATH,closed_state)
                print(f"{key}, loop {loop+1}: residual RMS {result['residual_rms']:.6f}",flush=True)
    for case in CASE_INDICES:
        i,j,t=case
        fig,ax=plt.subplots(layout='constrained')
        for n in counts:
            history=closed_state['runs'].get(f'{i},{j},{t},{n}',[])
            ax.plot(range(len(history)+1),[float(source['initial_rms'][i])]+[r['residual_rms'] for r in history],
                    marker='o',label='Zero start' if n==0 else f'{n} random + zero')
        ax.set(xlabel='AO loop',ylabel='Residual RMS [waves]',xticks=[0,1,2,3])
        ax.set_title(f"RMS {source['initial_rms'][i]:g}, reference SNR {source['reference_snr'][j]:g}, trial {t+1}")
        ax.legend()
        plt.show()
else:
    print('Closed-loop tests disabled. Enable RUN_CLOSED_LOOP after inspecting the replay results.')